# Feature 1: Transaction Parser

In [ ]:
# AI-assisted: Library import and notebook setup

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 20)

print("Libraries Imported Successfully")

Libraries Imported Successfully


In [ ]:
# AI-assisted: Load dataset

from google.colab import files

uploaded = files.upload()

Saving Data_set_for_DADS_June.csv to Data_set_for_DADS_June.csv


In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv("Data_set_for_DADS_June.csv")

In [ ]:
df.head()


,Date,Time,Description,Type,Amount,Balance,Mode,Ref
0,2024-01-01,03:11,AMAZON SELLER SVCS,Debit,₹2462,678275.0,UPI,TXN190872
1,01-Jan-24,05:44,BHIM-BMTC,DR,50.00,681007.0,UPI,TXN143064
2,01-Jan-24,09:35,NEFT-TECHCRUSH LABS-SALARY MAY24,CR,₹84728,484728.0,NEFT,TXN246316
3,2024-01-01,14:07,UPI-AMAN-8934@OKAXIS,Debit,₹1828,-748745.0,UPI,TXN569226
4,01 Jan 2024,14:23,BHIM-BLINKIT,Debit,270.00,680737.0,UPI,TXN968962


In [ ]:
df.sample(5)

,Date,Time,Description,Type,Amount,Balance,Mode,Ref
179,26/01/24,08:48,TUMMOC-BMTC,Debit,14.00,530692.0,UPI,TXN613261
23,2024-01-03,23:58,POS SWIGGY-RESTAURANT,Debit,₹238,666548.0,UPI,TXN975271
1156,2024-06-07,21:20,BUNDL TECH-INSTAMART,Debit,₹592,-201367.0,UPI,TXN246205
463,04/03/24,15:49,ANI Technologies,DR,371,329738.0,UPI,TXN957710
5,2024-01-01,14:48,BHIM ZEPTO,DR,Rs. 625,677650.0,UPI,TXN370902


In [ ]:
df.columns

Index(['Date', 'Time', 'Description', 'Type', 'Amount', 'Balance', 'Mode',
       'Ref'],
      dtype='object')

In [ ]:
df.dtypes

,0
Date,object
Time,object
Description,object
Type,object
Amount,object
Balance,float64
Mode,object
Ref,object


In [ ]:
df['Amount'] = (
    df['Amount']
      .astype(str)
      .str.replace('₹','', regex=False)
      .str.replace('Rs.','', regex=False)
      .str.replace(',','', regex=False)
      .str.strip()
)

df['Amount'] = pd.to_numeric(df['Amount'], errors='coerce')

In [ ]:
df['Amount'].head()

,Amount
0,2462.0
1,50.0
2,84728.0
3,1828.0
4,270.0


In [ ]:
df['Amount'].isnull().sum()

np.int64(0)

In [ ]:
df['Type'] = df['Type'].replace({
    'DR':'debit',
    'Debit':'debit',
    'CR':'credit',
    'Credit':'credit'
})

df['Type'] = df['Type'].str.lower()

In [ ]:
df['Type'].value_counts()

,count
Type,
debit,1322
credit,6


In [ ]:
print("Before:", df.shape)

df = df.drop_duplicates()

print("After :", df.shape)

Before: (1328, 8)
After : (1310, 8)


In [ ]:
df.isnull().sum()

,0
Date,0
Time,0
Description,0
Type,0
Amount,0
Balance,0
Mode,0
Ref,0


In [ ]:
df_clean = df.copy()

# Feature 2: Vendor Extractor

In [ ]:
df_clean['Description'].unique()

array(['AMAZON SELLER SVCS', 'BHIM-BMTC',
       'NEFT-TECHCRUSH LABS-SALARY MAY24', 'UPI-AMAN-8934@OKAXIS',
       'BHIM-BLINKIT', 'BHIM ZEPTO', 'UPI-UBER-2426@HDFCBANK',
       'POS SWIGGY BANGALORE', 'UPI-GROWWPAY@HDFCBANK', 'OLA ELECTRIC',
       'BMS MOVIE TICKETS', 'POS OLA-PRIME', 'SWIGGY-INSTAMART',
       'UPI-STARBUCKS@AXIS', 'UPI-THIRDWAVE@OKAXIS', 'ANI Technologies',
       'BMTC BUS PASS', 'POS TRUFFLES', 'FLIPKART INDIA',
       'POS SWIGGY-RESTAURANT', 'GROFERS INDIA P L', 'POS UBER BANGALORE',
       'BANGALORE ELEC SUPPLY', 'TWC INDIA', 'UPI-BESCOM-BILL@HDFCBANK',
       'UPI-AMAN-0816@OKAXIS', 'ROPPEN TRANSPORTATION', 'OLA CABS',
       'POS ZOMATO', 'UPI-AMAZONPAY@HDFCBANK', 'POS BLINKIT',
       'IMPS-RENT-LANDLORD-75500265', 'ZOMATO MEDIA P L',
       'UPI-ANKIT-6430@OKAXIS', 'UPI-OLACABS@HDFCBANK',
       'UPI-JIORECHARGE@PAYTM', 'UPI-CCD@HDFCBANK', 'Swiggy*Order',
       'INSTAMART BANGALORE', 'UPI-ZOMATO-LIMITED@PAYTM',
       'AVENUE SUPERMARTS', 'POS HP PETROL

In [ ]:
def extract_vendor(description):

    description = str(description).upper()

    for vendor, keywords in vendor_dict.items():

        for keyword in keywords:

            if keyword in description:
                return vendor

    return "Uncategorised"

In [ ]:
vendor_dict = {

    # Food Delivery
    "Swiggy": [
        "SWIGGY", "BUNDL", "SWIGGY*ORDER", "POS SWIGGY"
    ],

    "Zomato": [
        "ZOMATO", "ZOMATO MEDIA"
    ],

    # Quick Commerce
    "Blinkit": [
        "BLINKIT", "GROFERS"
    ],

    "Instamart": [
        "INSTAMART"
    ],

    "Zepto": [
        "ZEPTO", "KIRANAKART"
    ],

    "DMart": [
        "DMART", "AVENUE SUPERMARTS"
    ],

    # Shopping
    "Amazon": [
        "AMAZON", "AMZN"
    ],

    "Flipkart": [
        "FKART", "FLIPKART"
    ],

    "Nykaa": [
        "FSN", "NYKAA"
    ],

    # Cafes
    "Starbucks": [
        "STARBUCKS", "TATA STARBUCKS"
    ],

    "Third Wave Coffee": [
        "THIRDWAVE", "THIRD WAVE", "TWC INDIA"
    ],

    "Cafe Coffee Day": [
        "CCD", "COFFEE DAY"
    ],

    # Restaurants
    "Empire Restaurant": [
        "EMPIRE"
    ],

    "Meghana Foods": [
        "MEGHANA"
    ],

    "Truffles": [
        "TRUFFLES"
    ],

    "Restaurant": [
        "RESTAURANT", "DINEOUT"
    ],

    # Transport
    "Uber": [
        "UBER", "ANI TECHNOLOGIES"
    ],

    "Rapido": [
        "RAPIDO", "ROPPEN"
    ],

    "BMTC": [
        "BMTC", "TUMMOC"
    ],

    # Fuel
    "Indian Oil": [
        "IOC", "INDIAN OIL"
    ],

    "HP Petrol": [
        "HP PETROL"
    ],

    "BPCL": [
        "BPCL"
    ],

    # Entertainment
    "BookMyShow": [
        "BMS", "BIGTREE"
    ],

    # Utilities
    "BESCOM": [
        "BESCOM", "ELEC"
    ],

    "BWSSB": [
        "BWSSB"
    ],

    "VI": [
        "VI", "VODAFONE"
    ],

    # Salary
    "Salary": [
        "SALARY"
    ],

    # Rent
    "Rent": [
        "LANDLORD", "RENT"
    ],

    # ATM
    "Cash Withdrawal": [
        "ATM-WDL", "ATM"
    ],

    # Personal Transfers
    "P2P Transfer": [
        "AMAN", "NEHA", "SNEHA",
        "KARAN", "VIKAS"
    ]
}

In [ ]:
def extract_vendor(description):

    description = str(description).upper()

    for vendor, keywords in vendor_dict.items():

        for keyword in keywords:

            if keyword in description:
                return vendor

    return "Uncategorised"

In [ ]:
df_clean["vendor_clean"] = df_clean["Description"].apply(extract_vendor)

In [ ]:
df_clean[["Description","vendor_clean"]].head(20)

,Description,vendor_clean
0,AMAZON SELLER SVCS,Amazon
1,BHIM-BMTC,BMTC
2,NEFT-TECHCRUSH LABS-SALARY MAY24,Salary
3,UPI-AMAN-8934@OKAXIS,P2P Transfer
4,BHIM-BLINKIT,Blinkit
5,BHIM ZEPTO,Zepto
6,UPI-UBER-2426@HDFCBANK,Uber
7,BHIM-BLINKIT,Blinkit
8,POS SWIGGY BANGALORE,Swiggy
9,UPI-GROWWPAY@HDFCBANK,Uncategorised


In [ ]:
df_clean["vendor_clean"].value_counts()

,count
vendor_clean,
Swiggy,223
Uncategorised,166
Zomato,121
Uber,89
Amazon,86
...,...
BWSSB,8
Salary,6
P2P Transfer,6


In [ ]:
df_clean[df_clean["vendor_clean"]=="Uncategorised"]["Description"].value_counts()

,count
Description,
OLA CABS,21
POS OLA-PRIME,18
UPI-OLACABS@HDFCBANK,10
UPI-MYNTRA@HDFCBANK,10
INNOVATIVE RETAIL,8
...,...
UPI-PRIYA-0104@OKAXIS,1
ZERODHA SECURITIES,1
UPI-ANKIT-5946@OKAXIS,1


# Feature 3 – Category Tagger

In [ ]:
category_dict = {

    # Food Delivery
    "Swiggy": "Food Delivery",
    "Zomato": "Food Delivery",

    # Quick Commerce
    "Blinkit": "Quick Commerce",
    "Instamart": "Quick Commerce",
    "Zepto": "Quick Commerce",

    # Groceries
    "DMart": "Groceries",
    "Reliance Fresh": "Groceries",

    # Cafe
    "Starbucks": "Cafe",
    "Third Wave Coffee": "Cafe",
    "Cafe Coffee Day": "Cafe",

    # Restaurants
    "Empire Restaurant": "Restaurants",
    "Meghana Foods": "Restaurants",
    "Truffles": "Restaurants",
    "Restaurant": "Restaurants",

    # E-commerce
    "Amazon": "E-commerce",
    "Flipkart": "E-commerce",
    "Myntra": "E-commerce",
    "Nykaa": "E-commerce",

    # Transport
    "Uber": "Transport",
    "Ola": "Transport",
    "Rapido": "Transport",
    "BMTC": "Transport",

    # Fuel
    "Indian Oil": "Fuel",
    "HP Petrol": "Fuel",
    "BPCL": "Fuel",

    # Entertainment
    "BookMyShow": "Entertainment",

    # Utilities
    "BESCOM": "Utilities",
    "BWSSB": "Utilities",
    "VI": "Utilities",

    # Investments
    "Zerodha": "Investments",

    # Salary
    "Salary": "Income",

    # Rent
    "Rent": "Rent",

    # Cash
    "Cash Withdrawal": "Cash Withdrawal",

    # Personal
    "P2P Transfer": "Personal Transfer"
}

In [ ]:
df_clean["category"] = df_clean["vendor_clean"].map(category_dict)

In [ ]:
df_clean["category"] = df_clean["category"].fillna("Others")

In [ ]:
df_clean["category"].value_counts()

,count
category,
Food Delivery,344
Transport,181
Others,166
E-commerce,152
Quick Commerce,146
Cafe,99
Restaurants,73
Utilities,55
Fuel,28


In [ ]:
df_clean[["vendor_clean", "category"]].head(20)

,vendor_clean,category
0,Amazon,E-commerce
1,BMTC,Transport
2,Salary,Income
3,P2P Transfer,Personal Transfer
4,Blinkit,Quick Commerce
5,Zepto,Quick Commerce
6,Uber,Transport
7,Blinkit,Quick Commerce
8,Swiggy,Food Delivery
9,Uncategorised,Others


In [ ]:
df_clean.head()

,Date,Time,Description,Type,Amount,Balance,Mode,Ref,vendor_clean,category
0,2024-01-01,03:11,AMAZON SELLER SVCS,debit,2462.0,678275.0,UPI,TXN190872,Amazon,E-commerce
1,01-Jan-24,05:44,BHIM-BMTC,debit,50.0,681007.0,UPI,TXN143064,BMTC,Transport
2,01-Jan-24,09:35,NEFT-TECHCRUSH LABS-SALARY MAY24,credit,84728.0,484728.0,NEFT,TXN246316,Salary,Income
3,2024-01-01,14:07,UPI-AMAN-8934@OKAXIS,debit,1828.0,-748745.0,UPI,TXN569226,P2P Transfer,Personal Transfer
4,01 Jan 2024,14:23,BHIM-BLINKIT,debit,270.0,680737.0,UPI,TXN968962,Blinkit,Quick Commerce


# Feature 4 - Spending Overview

In [ ]:
total_credits = df_clean[df_clean["Type"]=="credit"]["Amount"].sum()

print("Total Credits : ₹{:,.2f}".format(total_credits))

Total Credits : ₹509,774.00


In [ ]:
total_debits = df_clean[df_clean["Type"]=="debit"]["Amount"].sum()

print("Total Debits : ₹{:,.2f}".format(total_debits))

Total Debits : ₹1,678,901.00


In [ ]:
net_savings = total_credits - total_debits

print("Net Savings : ₹{:,.2f}".format(net_savings))

Net Savings : ₹-1,169,127.00


In [ ]:
savings_rate = (net_savings / total_credits) * 100

print("Savings Rate : {:.2f}%".format(savings_rate))

Savings Rate : -229.34%


In [ ]:
print("Total Transactions :", len(df_clean))

Total Transactions : 1310


In [ ]:
print("Unique Vendors :", df_clean["vendor_clean"].nunique())

Unique Vendors : 31


In [ ]:
top_categories = (
    df_clean[df_clean["Type"]=="debit"]
    .groupby("category")["Amount"]
    .sum()
    .sort_values(ascending=False)
)

top_categories.head()

,Amount
category,
E-commerce,534348.0
Others,391328.0
Food Delivery,150839.0
Restaurants,117737.0
Rent,108000.0


In [ ]:
top_vendors = (
    df_clean[df_clean["Type"]=="debit"]
    .groupby("vendor_clean")["Amount"]
    .sum()
    .sort_values(ascending=False)
)

top_vendors.head()

,Amount
vendor_clean,
Uncategorised,391328.0
Amazon,328530.0
Flipkart,177510.0
Rent,108000.0
Swiggy,95523.0


In [ ]:
df_clean["vendor_clean"].value_counts().head()

,count
vendor_clean,
Swiggy,223
Uncategorised,166
Zomato,121
Uber,89
Amazon,86


In [ ]:
print("="*60)
print("SPENDDNA - EXECUTIVE SUMMARY")
print("="*60)

print(f"Total Credits      : ₹{total_credits:,.2f}")
print(f"Total Debits       : ₹{total_debits:,.2f}")
print(f"Net Savings        : ₹{net_savings:,.2f}")
print(f"Savings Rate       : {savings_rate:.2f}%")
print(f"Transactions       : {len(df_clean)}")
print(f"Unique Vendors     : {df_clean['vendor_clean'].nunique()}")

print("="*60)

print("\nTop 5 Categories")
print(top_categories.head())

print("\nTop 5 Vendors")
print(top_vendors.head())

SPENDDNA - EXECUTIVE SUMMARY
Total Credits      : ₹509,774.00
Total Debits       : ₹1,678,901.00
Net Savings        : ₹-1,169,127.00
Savings Rate       : -229.34%
Transactions       : 1310
Unique Vendors     : 31

Top 5 Categories
category
E-commerce       534348.0
Others           391328.0
Food Delivery    150839.0
Restaurants      117737.0
Rent             108000.0
Name: Amount, dtype: float64

Top 5 Vendors
vendor_clean
Uncategorised    391328.0
Amazon           328530.0
Flipkart         177510.0
Rent             108000.0
Swiggy            95523.0
Name: Amount, dtype: float64


# Feature 5 - Monthly Trend Analysis

In [ ]:
df_clean["Date"].dtype

dtype('O')

In [ ]:
df_clean["Date"] = pd.to_datetime(
    df_clean["Date"],
    errors="coerce",
    dayfirst=True
)

In [ ]:
df_clean["Date"].dtype

dtype('<M8[ns]')

In [ ]:
df_clean["month"] = df_clean["Date"].dt.month

In [ ]:
print(df_clean["Date"].dtype)

datetime64[ns]


In [ ]:
print(df_clean[["Date"]].head())

        Date
0 2024-01-01
1        NaT
2        NaT
3 2024-01-01
4        NaT


In [ ]:
df_clean["month"] = df_clean["Date"].dt.month

In [ ]:
df_clean["Date"] = pd.to_datetime(
    df_clean["Date"],
    errors="coerce",
    dayfirst=True
)

In [ ]:
df_clean["Date"].dtype

dtype('<M8[ns]')

In [ ]:
df_clean["Date"].head(10)

,Date
0,2024-01-01
1,NaT
2,NaT
3,2024-01-01
4,NaT
5,2024-01-01
6,2024-01-01
7,NaT
8,2024-02-01
9,NaT


In [ ]:
df_clean.columns

Index(['Date', 'Time', 'Description', 'Type', 'Amount', 'Balance', 'Mode',
       'Ref', 'vendor_clean', 'category', 'month'],
      dtype='object')

# Feature 6 - Time-of-Day Analysis

In [ ]:
df_clean["hour"] = df_clean["Time"].str[:2].astype(int)

In [ ]:
df_clean[["Time","hour"]].head()

,Time,hour
0,03:11,3
1,05:44,5
2,09:35,9
3,14:07,14
4,14:23,14


In [ ]:
hourly_spending = (
    df_clean[df_clean["Type"]=="debit"]
    .groupby("hour")["Amount"]
    .sum()
    .sort_index()
)

hourly_spending

,Amount
hour,
0,37969.0
1,38882.0
2,20283.0
3,34868.0
4,43901.0
...,...
19,67848.0
20,138089.0
21,81545.0


In [ ]:
peak_hour = hourly_spending.idxmax()

peak_amount = hourly_spending.max()

print("Peak Spending Hour :", peak_hour)
print("Amount Spent : ₹{:,.2f}".format(peak_amount))

Peak Spending Hour : 10
Amount Spent : ₹140,526.00


In [ ]:
late_night = df_clean[
    (df_clean["hour"] >= 21) &
    (df_clean["Type"] == "debit")
]

print("Late Night Transactions :", len(late_night))
print("Late Night Spending : ₹{:,.2f}".format(late_night["Amount"].sum()))

Late Night Transactions : 163
Late Night Spending : ₹185,815.00


In [ ]:
late_food = df_clean[
    (df_clean["hour"] >= 21) &
    (df_clean["category"] == "Food Delivery")
]

print(late_food[["Description", "Amount", "Time"]])

                      Description  Amount   Time
23          POS SWIGGY-RESTAURANT   238.0  23:58
39                     POS ZOMATO   695.0  21:30
64                 BUNDL Tech P L   238.0  21:23
141      UPI-SWIGGY-9474@HDFCBANK   181.0  23:53
151      UPI-SWIGGY-2954@HDFCBANK   254.0  21:47
...                           ...     ...    ...
1198  UPI-SWIGGY-INSTAMART@OKAXIS   225.0  23:36
1211                ZOMATO-DINING   632.0  21:37
1219                 Swiggy*Order   425.0  22:55
1250               BUNDL Tech P L   404.0  21:57
1313               BUNDL Tech P L   589.0  21:33

[61 rows x 3 columns]


In [ ]:
hour_category = pd.pivot_table(
    df_clean[df_clean["Type"]=="debit"],
    values="Amount",
    index="hour",
    columns="category",
    aggfunc="sum",
    fill_value=0
)

hour_category

category,Cafe,Cash Withdrawal,E-commerce,Entertainment,Food Delivery,Fuel,Groceries,Others,Personal Transfer,Quick Commerce,Rent,Restaurants,Transport,Utilities
hour,,,,,,,,,,,,,,
0,872.0,0.0,18650.0,0.0,832.0,2675.0,3895.0,6393.0,0.0,1542.0,0.0,1411.0,60.0,1639.0
1,549.0,0.0,9801.0,620.0,3017.0,2048.0,1536.0,16038.0,0.0,1402.0,0.0,960.0,0.0,2911.0
2,0.0,0.0,10246.0,0.0,928.0,0.0,1717.0,1930.0,0.0,1143.0,0.0,1336.0,358.0,2625.0
3,316.0,0.0,11839.0,0.0,1702.0,676.0,7080.0,10039.0,0.0,1165.0,0.0,1714.0,337.0,0.0
4,0.0,0.0,11365.0,0.0,2618.0,2010.0,1169.0,24646.0,0.0,768.0,0.0,0.0,103.0,1222.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19,584.0,0.0,10990.0,417.0,17300.0,8351.0,1248.0,10191.0,0.0,6992.0,0.0,8615.0,3160.0,0.0
20,1491.0,1000.0,40445.0,474.0,18372.0,705.0,2566.0,29942.0,0.0,7319.0,0.0,27663.0,3314.0,4798.0
21,0.0,1000.0,31809.0,0.0,12123.0,11845.0,1891.0,6468.0,266.0,6701.0,0.0,6348.0,1436.0,1658.0


In [ ]:
df_clean.columns

Index(['Date', 'Time', 'Description', 'Type', 'Amount', 'Balance', 'Mode',
       'Ref', 'vendor_clean', 'category', 'month', 'hour'],
      dtype='object')

# Feature 7 - Anomaly Detection

In [ ]:
category_mean = df_clean.groupby("category")["Amount"].transform("mean")
category_mean.head()

,Amount
0,3515.447368
1,200.585635
2,84962.333333
3,1807.000000
4,506.041096


In [ ]:
category_std = df_clean.groupby("category")["Amount"].transform("std")
category_std.head()

,Amount
0,4657.251378
1,153.918375
2,373.432011
3,791.690344
4,201.953422


In [ ]:
df_clean["z_score"] = (
    (df_clean["Amount"] - category_mean)
    / category_std
)

df_clean["z_score"] = df_clean["z_score"].round(2)

df_clean.head()

,Date,Time,Description,Type,Amount,Balance,Mode,Ref,vendor_clean,category,month,hour,z_score
0,2024-01-01,03:11,AMAZON SELLER SVCS,debit,2462.0,678275.0,UPI,TXN190872,Amazon,E-commerce,1.0,3,-0.23
1,NaT,05:44,BHIM-BMTC,debit,50.0,681007.0,UPI,TXN143064,BMTC,Transport,NaN,5,-0.98
2,NaT,09:35,NEFT-TECHCRUSH LABS-SALARY MAY24,credit,84728.0,484728.0,NEFT,TXN246316,Salary,Income,NaN,9,-0.63
3,2024-01-01,14:07,UPI-AMAN-8934@OKAXIS,debit,1828.0,-748745.0,UPI,TXN569226,P2P Transfer,Personal Transfer,1.0,14,0.03
4,NaT,14:23,BHIM-BLINKIT,debit,270.0,680737.0,UPI,TXN968962,Blinkit,Quick Commerce,NaN,14,-1.17


In [ ]:
anomalies = df_clean[df_clean["z_score"] > 3]

anomalies

,Date,Time,Description,Type,Amount,Balance,Mode,Ref,vendor_clean,category,month,hour,z_score
49,NaT,08:38,POS BANGALORE RESTAURANT,debit,7314.0,-411932.0,UPI,TXN388441,Restaurant,Restaurants,NaN,8,3.27
269,NaT,17:28,UPI-AMAZONPAY@HDFCBANK,debit,21986.0,-360151.0,UPI,TXN988380,Amazon,E-commerce,NaN,17,3.97
414,NaT,20:09,POS BANGALORE RESTAURANT,debit,8383.0,-567033.0,UPI,TXN308266,Restaurant,Restaurants,NaN,20,3.88
450,NaT,11:58,AMAZON IN,debit,18273.0,-558650.0,UPI,TXN228115,Amazon,E-commerce,NaN,11,3.17
464,NaT,16:12,POS TRUFFLES,debit,7441.0,-472880.0,UPI,TXN830484,Truffles,Restaurants,NaN,16,3.34
475,NaT,21:33,UPI-AMAZONPAY@HDFCBANK,debit,19917.0,-465439.0,UPI,TXN674071,Amazon,E-commerce,NaN,21,3.52
653,NaT,20:01,POS MEGHANA FOODS,debit,7931.0,-445522.0,UPI,TXN149269,Meghana Foods,Restaurants,NaN,20,3.63
1079,NaT,14:46,UPI-FLIPKART@HDFCBANK,debit,17831.0,-437591.0,UPI,TXN685380,Flipkart,E-commerce,NaN,14,3.07
1271,NaT,12:46,POS DINEOUT,debit,7935.0,-480815.0,UPI,TXN540191,Restaurant,Restaurants,NaN,12,3.63
1298,NaT,09:07,AMAZONIN MARKETPLACE,debit,22008.0,-699028.0,UPI,TXN313051,Amazon,E-commerce,NaN,9,3.97


In [ ]:
anomalies = anomalies.sort_values(
    by="z_score",
    ascending=False
)

anomalies.head(10)

,Date,Time,Description,Type,Amount,Balance,Mode,Ref,vendor_clean,category,month,hour,z_score
269,NaT,17:28,UPI-AMAZONPAY@HDFCBANK,debit,21986.0,-360151.0,UPI,TXN988380,Amazon,E-commerce,NaN,17,3.97
1298,NaT,09:07,AMAZONIN MARKETPLACE,debit,22008.0,-699028.0,UPI,TXN313051,Amazon,E-commerce,NaN,9,3.97
414,NaT,20:09,POS BANGALORE RESTAURANT,debit,8383.0,-567033.0,UPI,TXN308266,Restaurant,Restaurants,NaN,20,3.88
653,NaT,20:01,POS MEGHANA FOODS,debit,7931.0,-445522.0,UPI,TXN149269,Meghana Foods,Restaurants,NaN,20,3.63
1271,NaT,12:46,POS DINEOUT,debit,7935.0,-480815.0,UPI,TXN540191,Restaurant,Restaurants,NaN,12,3.63
475,NaT,21:33,UPI-AMAZONPAY@HDFCBANK,debit,19917.0,-465439.0,UPI,TXN674071,Amazon,E-commerce,NaN,21,3.52
464,NaT,16:12,POS TRUFFLES,debit,7441.0,-472880.0,UPI,TXN830484,Truffles,Restaurants,NaN,16,3.34
49,NaT,08:38,POS BANGALORE RESTAURANT,debit,7314.0,-411932.0,UPI,TXN388441,Restaurant,Restaurants,NaN,8,3.27
450,NaT,11:58,AMAZON IN,debit,18273.0,-558650.0,UPI,TXN228115,Amazon,E-commerce,NaN,11,3.17
1079,NaT,14:46,UPI-FLIPKART@HDFCBANK,debit,17831.0,-437591.0,UPI,TXN685380,Flipkart,E-commerce,NaN,14,3.07


In [ ]:
print("="*60)
print("ANOMALY DETECTION REPORT")
print("="*60)

print("Total Anomalies :", len(anomalies))

print("\nTop 10 Anomalies")

print(
    anomalies[
        [
            "Date",
            "Description",
            "vendor_clean",
            "category",
            "Amount",
            "z_score"
        ]
    ].head(10)
)

ANOMALY DETECTION REPORT
Total Anomalies : 10

Top 10 Anomalies
     Date               Description   vendor_clean     category   Amount  \
269   NaT    UPI-AMAZONPAY@HDFCBANK         Amazon   E-commerce  21986.0   
1298  NaT      AMAZONIN MARKETPLACE         Amazon   E-commerce  22008.0   
414   NaT  POS BANGALORE RESTAURANT     Restaurant  Restaurants   8383.0   
653   NaT         POS MEGHANA FOODS  Meghana Foods  Restaurants   7931.0   
1271  NaT               POS DINEOUT     Restaurant  Restaurants   7935.0   
475   NaT    UPI-AMAZONPAY@HDFCBANK         Amazon   E-commerce  19917.0   
464   NaT              POS TRUFFLES       Truffles  Restaurants   7441.0   
49    NaT  POS BANGALORE RESTAURANT     Restaurant  Restaurants   7314.0   
450   NaT                 AMAZON IN         Amazon   E-commerce  18273.0   
1079  NaT     UPI-FLIPKART@HDFCBANK       Flipkart   E-commerce  17831.0   

      z_score  
269      3.97  
1298     3.97  
414      3.88  
653      3.63  
1271     3.63  
475

# Feature 8 - Spending Archetype Detection

In [ ]:
category_spend = (
    df_clean[df_clean["Type"] == "debit"]
    .groupby("category")["Amount"]
    .sum()
)

category_spend

,Amount
category,
Cafe,31445.0
Cash Withdrawal,45500.0
E-commerce,534348.0
Entertainment,5601.0
Food Delivery,150839.0
Fuel,89303.0
Groceries,37463.0
Others,391328.0
Personal Transfer,10842.0


In [ ]:
total_spend = category_spend.sum()

print("Total Debit Spend : ₹{:,.2f}".format(total_spend))

Total Debit Spend : ₹1,678,901.00


In [ ]:
category_percent = (
    category_spend / total_spend
) * 100

category_percent = category_percent.round(2)

category_percent

,Amount
category,
Cafe,1.87
Cash Withdrawal,2.71
E-commerce,31.83
Entertainment,0.33
Food Delivery,8.98
Fuel,5.32
Groceries,2.23
Others,23.31
Personal Transfer,0.65


In [ ]:
archetypes = []

if category_percent.get("Food Delivery", 0) > 20:
    archetypes.append("🍔 THE FOODIE")

if category_percent.get("Quick Commerce", 0) > 15:
    archetypes.append("⚡ THE QUICK COMMERCE USER")

if category_percent.get("E-commerce", 0) > 15:
    archetypes.append("🛍️ THE SHOPAHOLIC")

if category_percent.get("Investments", 0) > 10:
    archetypes.append("📈 THE INVESTOR")

if category_percent.get("Transport", 0) > 12:
    archetypes.append("🚕 THE TRAVELLER")

if category_percent.get("Cafe", 0) > 8:
    archetypes.append("☕ THE CAFE LOVER")

if category_percent.get("Restaurants", 0) > 10:
    archetypes.append("🍽️ THE RESTAURANT EXPLORER")

if category_percent.get("Entertainment", 0) > 5:
    archetypes.append("🎬 THE ENTERTAINER")

In [ ]:
food_orders = df_clean[df_clean["category"] == "Food Delivery"]

late_food = food_orders[food_orders["hour"] >= 21]

if len(food_orders) > 0:
    late_percent = (len(late_food) / len(food_orders)) * 100
else:
    late_percent = 0

print("Late Night Food Orders :", round(late_percent,2), "%")

if late_percent > 50:
    archetypes.append("🌙 THE LATE NIGHT SNACKER")

Late Night Food Orders : 17.73 %


In [ ]:
if net_savings < 0:
    archetypes.append("💸 THE YOLO SPENDER")

In [ ]:
  print("="*60)
print("SPENDING ARCHETYPES")
print("="*60)

for item in archetypes:
    print(item)

SPENDING ARCHETYPES
🛍️ THE SHOPAHOLIC
💸 THE YOLO SPENDER


# Final SpendDNA Report

In [ ]:
print("="*70)
print("                SPENDDNA REPORT")
print("="*70)

print(f"Total Credits      : ₹{total_credits:,.2f}")
print(f"Total Debits       : ₹{total_debits:,.2f}")
print(f"Net Savings        : ₹{net_savings:,.2f}")
print(f"Savings Rate       : {savings_rate:.2f}%")
print(f"Transactions       : {len(df_clean)}")
print(f"Unique Vendors     : {df_clean['vendor_clean'].nunique()}")

print("\nTOP 5 CATEGORIES")
print(top_categories.head())

print("\nTOP 5 VENDORS")
print(top_vendors.head())

print("\nSPENDING ARCHETYPES")
for item in archetypes:
    print(item)

print("\nTOP ANOMALIES")
print(
    anomalies[
        ["Description", "Amount", "z_score"]
    ].head(5)
)

print("="*70)
print("END OF REPORT")
print("="*70)

                SPENDDNA REPORT
Total Credits      : ₹509,774.00
Total Debits       : ₹1,678,901.00
Net Savings        : ₹-1,169,127.00
Savings Rate       : -229.34%
Transactions       : 1310
Unique Vendors     : 31

TOP 5 CATEGORIES
category
E-commerce       534348.0
Others           391328.0
Food Delivery    150839.0
Restaurants      117737.0
Rent             108000.0
Name: Amount, dtype: float64

TOP 5 VENDORS
vendor_clean
Uncategorised    391328.0
Amazon           328530.0
Flipkart         177510.0
Rent             108000.0
Swiggy            95523.0
Name: Amount, dtype: float64

SPENDING ARCHETYPES
🛍️ THE SHOPAHOLIC
💸 THE YOLO SPENDER

TOP ANOMALIES
                   Description   Amount  z_score
269     UPI-AMAZONPAY@HDFCBANK  21986.0     3.97
1298      AMAZONIN MARKETPLACE  22008.0     3.97
414   POS BANGALORE RESTAURANT   8383.0     3.88
653          POS MEGHANA FOODS   7931.0     3.63
1271               POS DINEOUT   7935.0     3.63
END OF REPORT
